In [1]:
import os
import requests
import pandas as pd
from datetime import datetime
import pytz

# === Timezone Setup ===
eastern = pytz.timezone('US/Eastern')
today = datetime.now(eastern).date()
today_str = today.isoformat()

# === Helpers for safe conversion ===
def safe_float(x):
    try:
        return float(x)
    except (ValueError, TypeError):
        return 0.0

def safe_int(x):
    try:
        return int(x)
    except (ValueError, TypeError):
        return 0

# === Fetch Pitcher Stats ===
def get_pitcher_stats(player_id, season):
    url = f"https://statsapi.mlb.com/api/v1/people/{player_id}/stats"
    params = {
        "stats": "season",
        "season": season,
        "group": "pitching",
        "gameType": "R"
    }
    try:
        r = requests.get(url, params=params)
        r.raise_for_status()
        data = r.json()
        splits = data.get("stats", [])[0].get("splits", [])
        stat = splits[0].get("stat", {}) if splits else {}
        return {
            "ERA": safe_float(stat.get("era")),
            "WHIP": safe_float(stat.get("whip")),
            "W": safe_int(stat.get("wins")),
            "SO": safe_int(stat.get("strikeOuts")),
            "IP": safe_float(stat.get("inningsPitched")),
            "AVG": stat.get("avg")
        }
    except:
        return {}

# === Fetch Team Pitching Stats ===
def get_team_pitching_stats(team_id, season):
    url = f"https://statsapi.mlb.com/api/v1/teams/{team_id}/stats"
    params = {
        "stats": "season",
        "season": season,
        "group": "pitching"
    }
    try:
        r = requests.get(url, params=params)
        r.raise_for_status()
        data = r.json()
        splits = data.get("stats", [])[0].get("splits", [])
        stat = splits[0].get("stat", {}) if splits else {}
        return {
            "ERA": safe_float(stat.get("era")),
            "WHIP": safe_float(stat.get("whip")),
            "SO": safe_int(stat.get("strikeOuts")),
            "IP": safe_float(stat.get("inningsPitched")),
            "AVG": stat.get("avg")
        }
    except:
        return {}

# === Estimate Bullpen Stats ===
def estimate_bullpen_stats(team_stats, starter_stats):
    team_ip = team_stats.get("IP", 0.0)
    starter_ip = starter_stats.get("IP", 0.0)
    bullpen_ip = max(team_ip - starter_ip, 0.1)

    team_era = team_stats.get("ERA", 0.0)
    starter_era = starter_stats.get("ERA", 0.0)
    team_whip = team_stats.get("WHIP", 0.0)
    starter_whip = starter_stats.get("WHIP", 0.0)

    team_so = team_stats.get("SO", 0)
    starter_so = starter_stats.get("SO", 0)

    return {
        "ERA": round((team_era * team_ip - starter_era * starter_ip) / bullpen_ip, 2),
        "WHIP": round((team_whip * team_ip - starter_whip * starter_ip) / bullpen_ip, 2),
        "SO": max(team_so - starter_so, 0),
        "IP": round(bullpen_ip, 1),
        "AVG": None
    }

# === Main Data Gathering ===
def fetch_starting_pitchers_with_bullpens(target_date):
    date_str = target_date.isoformat()
    url = f'https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={date_str}&hydrate=probablePitcher(note,stats,person)'

    r = requests.get(url)
    if r.status_code != 200:
        print(f"❌ API request failed: {r.status_code}")
        return None

    games = r.json().get('dates', [])[0].get('games', []) if r.json().get('dates') else []
    rows = []

    for game in games:
        home = game['teams']['home']
        away = game['teams']['away']

        home_team = home['team']['name']
        home_id = home['team']['id']
        home_pitcher = home.get('probablePitcher', {})
        home_name = home_pitcher.get('fullName', 'TBD')
        home_pid = home_pitcher.get('id')

        away_team = away['team']['name']
        away_id = away['team']['id']
        away_pitcher = away.get('probablePitcher', {})
        away_name = away_pitcher.get('fullName', 'TBD')
        away_pid = away_pitcher.get('id')

        # Fetch stats
        home_sp = get_pitcher_stats(home_pid, today.year) if home_pid else {}
        away_sp = get_pitcher_stats(away_pid, today.year) if away_pid else {}

        home_team_stats = get_team_pitching_stats(home_id, today.year)
        away_team_stats = get_team_pitching_stats(away_id, today.year)

        home_bp = estimate_bullpen_stats(home_team_stats, home_sp) if home_team_stats and home_sp else {}
        away_bp = estimate_bullpen_stats(away_team_stats, away_sp) if away_team_stats and away_sp else {}

        rows.append({
            "Date": date_str,
            "Home Team": home_team,
            "Home Starter": home_name,
            "Home ERA": home_sp.get("ERA"),
            "Home WHIP": home_sp.get("WHIP"),
            "Home IP": home_sp.get("IP"),
            "Home SO": home_sp.get("SO"),
            "Home AVG": home_sp.get("AVG"),
            "Home BP ERA": home_bp.get("ERA"),
            "Home BP WHIP": home_bp.get("WHIP"),
            "Home BP IP": home_bp.get("IP"),
            "Home BP SO": home_bp.get("SO"),

            "Away Team": away_team,
            "Away Starter": away_name,
            "Away ERA": away_sp.get("ERA"),
            "Away WHIP": away_sp.get("WHIP"),
            "Away IP": away_sp.get("IP"),
            "Away SO": away_sp.get("SO"),
            "Away AVG": away_sp.get("AVG"),
            "Away BP ERA": away_bp.get("ERA"),
            "Away BP WHIP": away_bp.get("WHIP"),
            "Away BP IP": away_bp.get("IP"),
            "Away BP SO": away_bp.get("SO"),
        })

    return pd.DataFrame(rows)

# === Execute and Save ===
df = fetch_starting_pitchers_with_bullpens(today)

if df is not None and not df.empty:
    pd.set_option("display.max_columns", None)
    print("✅ Starting pitchers and bullpen stats:")
    print(df.head())

    output_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "pitching_stats"))
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f"pitching_stats_{today_str}.csv")
    df.to_csv(output_file, index=False)
    print(f"✅ CSV saved to {output_file}")
else:
    print("⚠️ No data found.")


✅ Starting pitchers and bullpen stats:
         Date              Home Team    Home Starter  Home ERA  Home WHIP  \
0  2025-06-01          Texas Rangers    Jacob deGrom      2.34       0.98   
1  2025-06-01         Atlanta Braves     Bryce Elder      4.56       1.30   
2  2025-06-01      Baltimore Orioles  Charlie Morton      6.28       1.60   
3  2025-06-01  Philadelphia Phillies   Ranger Suárez      2.72       1.16   
4  2025-06-01      Toronto Blue Jays   Kevin Gausman      3.82       1.02   

   Home IP  Home SO Home AVG  Home BP ERA  Home BP WHIP  Home BP IP  \
0     69.1       66     .203         3.22          1.18       453.0   
1     49.1       37     .257         3.58          1.23       460.9   
2     53.0       52     .278         5.16          1.45       451.1   
3     36.1       33     .232         4.10          1.32       491.1   
4     70.2       68     .223         4.09          1.24       452.0   

   Home BP SO            Away Team     Away Starter  Away ERA  Away WHI